For det er muligt at lave markov kæderne, så skal der importers nogen bibloteker.

In [3]:
import re
import numpy as np
import random

In [ ]:

def read_file_into_tokens(file_path):
    """
    Læser en given fil, og omdanner det til en rå liste af tokens (altså individuelle ord)
    """

    # Her læser vi filens indhold
    file_contents = ""
    file = open(file_path, "r", encoding="utf-8")
    for line in file:
        file_contents += line
    file.close()
    
    # Vi laver alt indhold lower-case
    file_contents = file_contents.lower()
    
    # Her bruger vi et værktøj der hedder regex.
    # Det er bare en smart måde at vi hurtigt kan få omdannet vores fil-indhold til en liste af strings ved at opdele den ved specifikke tegn
    raw_tokens = re.split(r"(?![:,\.;\"][0-9])[:,\.;?\" \r\n]+", file_contents)
    
    # Behold kun de tokens som har en værdi (altså ingen tokens som er tom tekst)
    tokens = []
    for raw_token in raw_tokens:
        if raw_token:
            tokens.append(raw_token)
    return tokens

def create_states(data, amount_of_words=1):
    """
    Skaber individuelle states til vores data ud fra mængden af ord der betragtes på en gang  
    """
    list_of_states = []
    
    # Vi gå gennem vores data ud fra index idet vi skal kombinere dele af listen af gangen
    for i in range(len(data) + 1 - amount_of_words):
        # Kombiner fremliggende naboer op til mængden af betragtede ord (eg. amount_of_words = 2 medfører at hvert token bliver 2 ord lang)
        list_of_states.append(" ".join(data[i:i + amount_of_words]))

    return list_of_states


def generate_markov_chain(states):
    markov_chain={}
    """
    Genererer en dictionary som angiver mængden af forekomster og naboliggende ord (og deres hyppigheder) i vores datasæt som en 
    """
    for index, word in enumerate(states):
        # Positionen til det næste ord i vores liste
        nabo_index = index + 1

        # Hvis det næste ord i sekvensen ligger udenfor listen, er vi færdige.
        if nabo_index >= len(states):
            # Hvis det allersidste ord ikke findes andre steder end i slutningen, kan det som state i markov kæden ikke føre til andre states.
            # I dette tilfælde, siger vi derfor blot at dens næste ord er sig selv, således at vi ikke har en slags 'blind vej' i vores endelige markov kæde.
            if word not in markov_chain:
                markov_chain[word] = [1,{}]
                #markov_chain[word] = [1,{}]
                markov_chain[word][1][word] = 1
                markov_chain[word][1][next_word] = 1
            break
            
        next_word = states[nabo_index].split(" ")[-1]
        
        # Hvis vi har registreret vores ord i dictionary i forvejen, så tilføj 1 til hyppigheden og forekomsterne
        if word in markov_chain:
            # Ved 0 indekset ligger totale mængde forekomster i teksten
            markov_chain[word][0] += 1
            
            # Ved 1 indekset ligger en dictionary som viser hyppigheden for hvert naboliggende ord.
            # Opdater hyppigheden for det naboliggende ord 
            if next_word in markov_chain[word][1]:
                markov_chain[word][1][next_word] += 1
            else:
                markov_chain[word][1][next_word] = 1

        # Hvis ordet ikke er registreret endnu, registrerer vi den til vores data dictionary
        else:
            markov_chain[word] = [1,{}]
            markov_chain[word][1][next_word] = 1
    
    # Ved alle states sorterer vi de næste states ud fra deres hyppighed. 
    # Dette gør vi for at gøre det nemmere at beregne sandsynlighed
    for state in markov_chain.keys():
        markov_chain[state][1] = dict(sorted(markov_chain[state][1].items(), key = lambda item: item[1], reverse=True))

    return markov_chain


def calculate_state_chances(temperature, total_occurences, possible_next_states):
    """
    Beregner chancerne for at gå til hvert mulige state
    """
    return [calculate_single_state_chance(temperature, int(total_occurences), len(possible_next_states), state) for state in possible_next_states.values()]


def calculate_single_state_chance(temperature, total_occurances, possible_next_state_count, next_state_frequency):
    """
    Beregner en chance for at gå til hvert mulige state
    """
    
    if temperature<=1:
        # Denne beregning gør at temperatur på 0 medfører en determinisk generation, samt at ved 1 så generering chancen direkte ækvivalent med frekvensen / totale hyppighed.
        # Imellem disse 2 ender, er en glat overgang.
        return  next_state_frequency / ((1-temperature) * next_state_frequency + total_occurances * temperature)
    
    else:
        
        # Hvis temperature = 1, bliver den flippet til 0. Derefter beregnes
        """
        Først temperature = temperature-1 så temperature = 0
        I slutning skal det være next_state_frequency/totalt_occurences
        så next_state_frequency*(1-temperature)+temperature = next_state_frequency*(1-0)+0 = next_state_frequency
        så temperature = total_occurrences

        q= dif_amount*r+total_amount*(1-r) = dif_amount*0+total_amount*(1-0) =total_amount
        q = total_amount

        t/q =n/total_amount

        """
        
        
        # Hvis r=2 så bliver den 1 og det her er, hvad der skal ske med den.
        """
        Først r = r-1 så r = 1
        I slutning skal det være 1/dif_amount
        så n*(1-r)+r =n*(1-1)+1 = n*0+1 = 1
        så t = 1

        q= dif_amount*r+total_amount*(1-r) = dif_amount*1+total_amount*(1-1) =dif_amount
        q = dif_amount
        t/q= 1/dif_amount 
        """
        
        temperature = temperature-1
        t = next_state_frequency*(1-temperature)+temperature

        q = possible_next_state_count*temperature+total_occurances*(1-temperature) 
        return t/q
     

def generate_text(markov_chain, amount_to_generate = 10, starting_state = None, temperature = 1):
    """
    Genererer tekst ud fra den givne markov kæde. 
    Bemærk at hvis starting_state er None, bliver en tilfældig starting state valgt fra markov kæden.
    """
    
    # Hvis vi ikke har givet et start state, så vælg et tilfældigt
    if starting_state is None:
        starting_state = random.choice(markov_chain.keys())
    
    # Gør vores start til lower-case
    starting_state = starting_state.lower()

    # Tjek vores input
    if starting_state not in markov_chain.keys():
        raise ValueError("Start staten findes ikke som et state i markov kæden")
    if (amount_to_generate <= 0):
        raise ValueError("Du kan ikke generere nul eller en negativ mængde tekst")
    if (temperature < 0 or temperature > 2):
        raise ValueError("Temperatur skal være mellem 0 og 2")
    
    # Vi bestemmer mængden af ord betragtet på en gang ud fra det givne start ordet. 
    # Dette bør altid være den samme mængde som mængden af ord betragtet i selve markov kæden.
    amount_of_words = len(starting_state.split(" "))
    
    # Denne liste vil indeholde vores genererede tekst. Den starter derfor med at indeholde vores startord
    generated_text = starting_state.split(" ")
    
    # Vi bruger denne til at holde styr på hvad vi gerne vil generere det næste ud fra
    prev_state = starting_state
    
    # Generer tekst
    for i in range(amount_to_generate):
        total_occurances = markov_chain[prev_state][0]
        next_possible_states = markov_chain[prev_state][1]
        next_possible_state_chances = calculate_state_chances(temperature, total_occurances, next_possible_states)
        
        # Vi opsummere løbende chancerne for hvert state. Når den overgår vores tilfældigt-satte threshold, er det den state vi går til næst.
        # Dette er den simpleste måde at programmatisk vælge et element fra en liste tilfældigt ud fra en chance.
        random_threshold = np.random.rand(1,1)[0][0]
        sum_chance = 0

        # Variablen er, hvor meget chance der er tilbage før der har noget på, som bliver anvendt senere til en form for normalisering
        amount_left = 1

        # Variablen er hvad der skal ganges med, så det er normaliset på en måde for, at sikre at det altid bliver summed up til 1 i slutning
        multiply_to_norm = 1
        
        # Vi gennemgår hvert næste mulige state med et indeks, idet vi bruger seperate lister til chancerne og selve states.
        for next_state_index, next_state_text in enumerate(next_possible_states.keys()):
            
            # Tjek om vores tilfældige threshold er nået- hvis ja, er det nuværende element det tilfældigt valgte. Ellers, fortsæt gennem vores muligheder.
            sum_chance += next_possible_state_chances[next_state_index]*multiply_to_norm
            if sum_chance <= random_threshold:
                # Opdater mængden der er left baseret på sum_chancen
                amount_left = 1 - sum_chance 
                # dette bliver til hvad man skal gange alle værdierne med, så deres sum giver mængden der er tilbage for at sikre at der altid vil 1.
                multiply_to_norm = 1/(sum(next_possible_state_chances[next_state_index:])*(1/(amount_left)))
                continue

            # Vi har hermed vores næste state.
            generated_text.append(next_state_text)
            
            # Opdater det nuværende state til at være de sidste ord i vores genererede tekst
            prev_state = " ".join(generated_text[-amount_of_words:])
            break

    # Vores genererede tekst består af individuelle ord. Vi samler dem for at få et sammenhængende stykke tekst
    return " ".join(generated_text)


In [6]:
# Den første del før man kan anvende noget som helst omkring Markov kæder, er at skaffe og behandle dataen som din markov kæde bliver skabt ved.
# Det gør vi først ved at læse noget data omtil tokens
example_tokens = read_file_into_tokens(r"C:\programmering\UNF\KIC26\Markov-chain\DataTextFolder\test_data_file.txt")

# Efter vi har omdannet dataen til tokens, så skal vi skabe states ud fra dem.
example_states = create_states(example_tokens)
 
# Når states er lavet alt der, så er det bare selve makrov kæden der mangler at blive lavet.
example_markov_chain = generate_markov_chain(example_states)

"""
Nu er markov kæden skabt, så er den klar til at producere text.
Når man skal anvende den til nu at generere text, skal vi vælge en start state, den state i markov kæden vi starter vores generering ved.
Derudover skal vi også vælge en temperatur, en værdi som angiver tilfældigheden af hvert step.
En temperatur på 0 medfører her fuldstændig deterministiske steps, mens 1 er helt normal markov kæde, og 2 er fuldstændigt tilfældigt.
Start staten skal selvfølgelig være en af markov kædens stater.
"""
generate_text(markov_chain=example_markov_chain, amount_to_generate=15, starting_state="hej", temperature=1)

'hej mit navn er bob og jeg kan godt lide at spise is'

Som der kan ses, så er teksten der bliver genereret ikke alt for god. 
Det kan jo tænkes, at det skyldes at datasættet er meget lille, som medfører at markov kæden ikke har så meget at træne sig på.

Selvom vores output er dårligt lige nu, så er det stadig en metode til at generere tekst.

Med udgangspunkt i metoden vist ovenover, skal i kigge på forskellige måder at ændre output. 
I de følgende opgaver er der somså ikke specifikt et rigtig svar, bare forskellige løsningsmåder.

### Opgave 1)

Ligesom der er blevet gjort ovenover til at generere tekst, skal i nu opstille jeres egen markov kæde nu.
I kan blot gøre brug af de samme funktioner som bruges ovenover.

Vælg et datasæt fil `dataset`

Ud fra det valgte datasættet, opstil til `tokens`

Genererer derefter `states` ud fra `tokens`

ud fra `tokens`, lav en markov kæde `markov_chain`

In [ ]:
# Lav de forskellige trin her
dataset = "vælg en af de forskellige datasæt her"


Ud fra markov kæden; genererer tekst med forskellige starting states (tip: hvis du ikke ved hvilket starting state du vil bruge, kan du altid kigge i datasæt filens indhold- tænk på hvad hvert state er).

In [ ]:
# Din kode til at generere tekst her


Forsøg nu at skabe en funktion, som skaber en markov kæde ud fra en given fil.

In [ ]:
def create_markov_chain_from_file(file_path):
    # Implementer funktionen herunder
    

Anvend funktionen til at lave forskellige markov kæder ud fra forskellige filer.

Anvend de forskellige markov kæder du har lavet, og se hvordan dit output ændrer sig mellem dem.

In [ ]:
# Genererer tekst her


Vælg en af markov kæderne, som du lavet i opgave 0 til brug i de næste opgaver.

### Opgave 2)

Ændre randomness til værdier mellem 0 og 2, og se hvad det sker. 

Hvilken værdi er den 'bedste'?

Hvad sker der når den er 0?

Hvad sker der når den 2?

In [8]:
# Anvend generate_text med jeres markov kæde her få de forskellige svar


### Opgave 3)

I opgave 2 skulle i sætte randomnes til 0. Der kunne i se, at den ente med at loop sætninger efter et hvis stykke tid, men hvorfor? Prøv at lave en forklaring- diskuter evt. med sidemanden.

### Opgave 4)

I har set, hvad der sker hvis man ændrer temperaturen, men hvad hvis I i stedet ændrede formlen på hvordan selve sandsynlighederne beregnes?
Funktion `calculate_single_state_chance` er den funktion, som (more or less) bestemmer den besluttende sandsynlighed for at gå fra et state til et andet.

Prøv at ændre på `calculate_single_state_chance` og se hvad der sker med output. 

In [7]:
# Her er hvor i skal skrive jeres nye chance_formel 
"""
Funktionen herunder er chance funktionen som bestemmer chancen for at gå til hver mulige næste state.
Når i ændrer på den skal i ikke ændre på dens input argumenter, men der er ellers ingen begrænsning på hvad i har lyst til at bestemme temperaturen ved.
"""

def calculate_single_state_chance(temperature, total_occurances, possible_next_state_count, next_state_frequency):
    """
    Beregner chancen for at gå til et muligt state
    total_occurances angiver hvor mange gange det nuværende state opstår i datasættet
    possible_next_state_count angiver mængden af forskellige states den nuværende state forbinder videre til
    next_state_frequency angiver vægten af forbindelsen fra den nuværende state til den beregnede state. 
    """
    
    if temperature<=1:
        # Denne beregning gør at temperatur på 0 medfører en determinisk generation, samt at ved 1 så generering chancen direkte ækvivalent med frekvensen / totale hyppighed.
        # Imellem disse 2 ender, er en glat overgang.
        return  next_state_frequency / ((1-temperature) * next_state_frequency + total_occurances * temperature)
    
    else:
        
        # Hvis temperature = 1, bliver den flippet til 0. Derefter beregnes
        """
        Først temperature = temperature-1 så temperature = 0
        I slutning skal det være next_state_frequency/totalt_occurences
        så next_state_frequency*(1-temperature)+temperature = next_state_frequency*(1-0)+0 = next_state_frequency
        så temperature = total_occurrences

        q= dif_amount*r+total_amount*(1-r) = dif_amount*0+total_amount*(1-0) =total_amount
        q = total_amount

        t/q =n/total_amount

        """
        
        
        # Hvis r=2 så bliver den 1 og det her er, hvad der skal ske med den.
        """
        Først r = r-1 så r = 1
        I slutning skal det være 1/dif_amount
        så n*(1-r)+r =n*(1-1)+1 = n*0+1 = 1
        så t = 1

        q= dif_amount*r+total_amount*(1-r) = dif_amount*1+total_amount*(1-1) =dif_amount
        q = dif_amount
        t/q= 1/dif_amount 
        """
        
        temperature = temperature-1
        t = next_state_frequency*(1-temperature)+temperature

        q = possible_next_state_count*temperature+total_occurances*(1-temperature) 
        return t/q
     

Lav opgave 2 med den nye chance funktion

In [ ]:
# Her laver i opgave 1 med den nye funktion


### Opgave 5)

Filen `arrangoererne.txt` indeholder de samtlige arrønger tekster der står på hjemmesiden, sat op som en data fil. I skal nu lave en markov kæde ud fra denne fil, og derefter generere noget tekst ud fra den markov kæde.

In [ ]:
# Din kode herunder


Filen `arrangoerne_copy.txt` er det samme som `arrangoererne.txt`. Dit mål nu er at ændre i filen, således at når du genererer tekst med markov kæden, skal den som minimum sige "[jeg] er en arrangør på campen" (men du kan sagtens også se om du kan få den til at sige andre ting).

In [ ]:
# Din kode herunder


### Opgave 6)

Her skal du kigge på hvad der sker, når man anvender en markov kæde til at generere trænings data til sig selv.

Vælg en markov kæde, som du har lavet i en af de forrige opgaver.

Funktionen tager følgende argumenter:

**markov_chain:** Selve markov kæden <br>
**amount_of_words:** Antal ord den skal generere i filen<br>
**temperature:** Temperaturen der skal anvendes<br>
**amount_of_times:** Antal gange den iterativt laver ny data ud fra markov kæden<br>
**start_state:** Er start staten der bliver anvendt når den skal generere dataen<br>

Anvend funktionen til at lave en markov kæde ud fra den markov kæde du valgte tidligere, hvor amount_of_times = 1.



In [ ]:



# Funktion her tager en markov kæde, og itererer på den ved at den skabe ny text, generere en ny markov kæde ud fra den tekst, og så gentag
# Det er lidt ligesom de der "google translated ___ times" videoer
"""
amount_of_words er hvor mange ord der skal være i den nye fil
amount_of_times er antal af gange der laver ny data ud fra markov kæden, så amount_of_times = 1 så er det direkte fra den oprindlige markov kæde,
hvis amount_of_times = 2 så laver den dataen ud fra en markov kæde, som er baseret på den fil der kommer, hvis amount_of_times = 1
"""
def generate_iterated_markov_chain(markov_chain, amount_of_words=10000, temperature=1, amount_of_times=1, start_state = None):
    new_markov_chain = markov_chain
    
    # Funktion looper, så den iterativt laver en ny markov kæde efter hvert loop
    for i in range(amount_of_times):
        print("",f"Iteration {i}")
        
        # Vi genererer først ny text med den nuværende markov kæde
        text = generate_text(new_markov_chain, amount_of_words, start_state, temperature).split(" ")
        
        # Vi genererer nu en ny markov kæde med vores nyligt genererede tekst
        new_markov_chain = generate_markov_chain(create_states(text))

    return new_markov_chain

In [ ]:
# Lav markov kæden her


Når du har markov kæden, så prøv nu at generere tekst

In [ ]:
# Generer teksten her


Ændre kun på amount_of_words og se hvad der sker.

Ændre kun på temperature og se hvad der sker.

Ændre kun på start_state og se hvad der sker.

Ændre kun på amount_of_times og se hvad der sker.

Prøv nu at ændre på flere af argumenter på en gang og se hvad der sker med teksten.

In [ ]:
# Din kode herunder



Endtil videre har vi kun kigget på hvor start state var et ord, men der er intet der stopper os for at gøre så hver state består af mere end et ord. 
Når vi laver states, så skal man gøre så at hvert state består af n mængde ord i stedet for 1. I funktion create_states er der et input som hedder `amount_of_words`, som styrer hvor mange ord hver state består af. 

### Opgave 7)

Ændre på funktionen du lavede i opgave 2, så den nu også tager et extra argument, som angiver hvor mange ord der skal være på hvert state.

In [54]:
#Lav den nye funktion her

Vælg et dataset.

Lav en markov kæde ud fra datasettet, men med to ord som states og se, hvordan den generere tekst i forhold til en.

In [ ]:
# Generer markov kæderne og teksten her


Lav nu en markov kæde med 3 ord, og se hvordand en generere.

In [ ]:
# Generer markov kæderne og teksten her


### Opgave 8)

I har ændret på mange forskellige parameter, og fået forskellige resultater. Lav nu den bedste (efter egen holdning) markov kæde tekst generator du kan ved at ændre på de forskellige argumenter.

In [ ]:
# Din kode herunder

### Opgave 9)

I kommende blokke skal i arbejde med LLMer. Hvis i er nået hertil, kan i bruge denne sidste blok til at forsøge at skabe den bedste mulige markov kæde til generering af tekst.

Når i arbejder med LLMer, skal i anvende andre datasæt. Disse ligger under mappen DataTextHistorie. Anvend disse datasæt til at generere text, så i kan bruge det til senere sammenligning med tekstgenereringen gjort af en LLM.
Læg især mærke til den slags opførsler, både gode og dårlige, som opstår for hver slags textgenerering, og se om i kan sætte ord på hvorfor de opstår givet modellernes teoretiske grundlag.

In [ ]:
# Lav en markov kæde med den nye data her
